In [ ]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


# 03 · K-Means 客户聚类
> 目标：回答「会员除了 RFM 规则分群，还能不能被无监督地分成结构不同的群体」。
> 关键结论见文末：**轮廓系数达标与否取决于特征空间，不取决于调参**。

In [ ]:
import pandas as pd

from retailinsight.io.loader import read_parquet
from retailinsight.settings import INTERIM_DIR, MART_CLUSTER, MART_RFM

rfm = read_parquet(MART_RFM)
txn = read_parquet(INTERIM_DIR / "transactions.parquet")
items = read_parquet(INTERIM_DIR / "transaction_items.parquet")
products = read_parquet(INTERIM_DIR / "products.parquet")


## 3.1 候选特征空间
> `base6` = 计划 §8.1 规定的 6 维；`rfm3` = 只含对人群有区分度的核心维度；
> `rfm3_behavior` = 再叠加行为维度（周末占比 / 折扣敏感度 / 季节性强弱）。

In [ ]:
from retailinsight.features.cluster_features import CLUSTER_FEATURE_SPACES, build_cluster_features
from retailinsight.models.cluster import build_cluster_matrix, choose_k

feats = build_cluster_features(rfm, txn, items, products)
print(f"参与聚类的会员: {len(feats):,}（mart_rfm 共 {len(rfm):,}）")
for name, cols in CLUSTER_FEATURE_SPACES.items():
    print(f"  {name:16s} -> {cols}")


## 3.2 逐 K 评估：肘部法 + 轮廓系数
> 轮廓系数在 3.5 万样本上是 O(n²)，因此按固定种子抽样 5000 条计算。

In [ ]:
X, _ = build_cluster_matrix(feats, ["recency_days", "frequency", "monetary"])
metrics = choose_k(X)
pd.DataFrame(metrics).T.assign(
    min_share=lambda d: d["cluster_sizes"].apply(lambda s: min(s.values()) / sum(s.values()))
).round(4)


## 3.3 神谕上界检验：0.35 到底可达不可达？
> 直接用生成器埋的**真实 8 类人群**当标签算轮廓系数。
> 若神谕值也很低，说明该特征空间本身无法分离真实结构，再调参也无济于事。

In [ ]:
from retailinsight.models.cluster import oracle_silhouette

members = read_parquet(INTERIM_DIR / "members.parquet")
persona = members.set_index("member_id")["_persona"]
p = feats["member_id"].map(persona).values

for name, cols in CLUSTER_FEATURE_SPACES.items():
    Xs, _ = build_cluster_matrix(feats, cols)
    print(f"  {name:16s} 神谕上界 silhouette = {oracle_silhouette(Xs, p):.4f}")


## 3.4 各特征空间的完整评估与择优

In [ ]:
from retailinsight.models.cluster import evaluate_space, select_space

evals = [evaluate_space(feats, name, cols, persona) for name, cols in CLUSTER_FEATURE_SPACES.items()]
for e in evals:
    print(f"  {e['space']:16s} K={e['k']}  silhouette={e['silhouette']:.4f}  "
          f"ARI={e['ari']:.4f}  达标={e['reached_target']}  神谕={e['oracle_silhouette']:.4f}")

choice = select_space(evals, has_persona=True)
print("\n选中:", choice["selected_space"], "（择优准则:", choice["criterion"], "）")


## 3.5 簇画像与业务命名

In [ ]:
chosen = next(e for e in evals if e["space"] == choice["selected_space"])
chosen["desc"][["cluster_id", "cluster_name", "member_cnt", "member_share", "gmv_share",
                 "active_pct", "frequency_pct", "monetary_pct"]].round(3)


## 3.6 与先验人群的列联表（埋点闭环）
> 计划 §4.2 预埋的 8 类人群在这里终于有了消费方。

In [ ]:
cluster = read_parquet(MART_CLUSTER)
ct = pd.crosstab(cluster["cluster_name"],
                 cluster["member_id"].map(persona))
ct
